In [ ]:
import jax
import os

# # Avoid OOM errors
# os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"]=".66"
# # os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"]="false"
# print(f"JAX backend: {jax.default_backend()}")

If you don't set the GPU memory allocation, then you can accidentally run out of memory. Sometimes GPU usage is persistant because notebooks, I think.

In [ ]:
import sys
sys.path.append("../")

%load_ext autoreload
%autoreload 2

import pulse_simulator as ps

In [ ]:
from datetime import datetime
import functools
import itertools
import logging
import numpy as np
import pickle as pkl
import qiskit as qk
from qiskit import pulse
import qiskit_aer
import qiskit.providers.fake_provider as qk_fp
import qiskit_dynamics as qk_d
import qutip as qt
import re 
from scipy.integrate import simpson
from scipy.linalg import sqrtm, expm
from tqdm import tqdm
import warnings

import matplotlib.pyplot as plt

In [ ]:
Units = 1e9

# Undo units
GHz = 1 / Units
MHz = 1e3 / Units
μs = Units / 1e3
ns = Units

backend = qk_fp.FakeManila()
dt = backend.configuration().dt * ns
desired_1qb_duration = 225  # dt
desired_2qb_duration = 225 * 4  # dt

# Gate time in SI units (granularity restriction)
duration_1qb = ps.qiskit_closest_multiple_of_16(desired_1qb_duration * dt / ns, backend)
duration_2qb = ps.qiskit_closest_multiple_of_16(desired_2qb_duration * dt / ns, backend)

# Misc.

In [ ]:
mb = ps.MomentBuilder()

qc = qk.QuantumCircuit(2)
qc.sx(0)
qc.sx(1)
qc.barrier()
qc.ecr(0, 1)
qc.barrier()
qc.x(0)
qc.barrier()
qc.ecr(1, 0)
qc.barrier()
qc.x(1)

# expected = [
#     {0: 1.5707963267948966, 1: -3.141592653589793},
#     {0: "SX", 1: "SX"},
#     {1: -3.141592653589793},
#     {(0, 1): "ECR"},
#     {0: "X"},
# ]
mb.build_moments(qc, timed=True)

# Pulses

We now have a pair of identity gates.

In [ ]:
# Load shapes
fname = "../pico-pulses/saved-pulses-2024-02-16/a_single_qubit_gateset_R1e-5.csv"
knames = ["X1", "X2", "SX1", "SX2", "I1"]
pico_a = {k: v for k, v in zip(knames, ps.from_csv(fname))}
# pico_a["I2"] = np.zeros(pico_a["I1"].shape)

# The initial run takes awhile to construct the symbolic pulse shapes
x_gates = {}
for k, v in pico_a.items():
    x_gates[k] = ps.scalable_symbolic_pulse(duration_1qb, v, f"custom-{k}")

# Calibrate the robust X gates
ts = np.arange(0, dt * duration_1qb, dt)
fig, ax = plt.subplots(1, 1, figsize=[4,2])
cal_x_gates = {}
# Only hit the first 4 gates
cal_values = [1.5 * np.pi, 0.5 * np.pi, 1.25 * np.pi, 0.25 * np.pi]
cal_amps = []
for θ, (k, v) in zip(cal_values, x_gates.items()):
    amp = ps.get_amplitude_for_rotation(θ, ps.get_samples(v).real, dt)
    cal_amps.append(amp)
    print(f"Adjusting {k} to amplitude {amp}")
    cal_x_gates[k] = qk.pulse.ScalableSymbolicPulse(
        pulse_type=f"{k}-calibrated",
        duration=duration_1qb,
        amp=amp,
        angle=0.0,
        limit_amplitude=False,
        envelope=v.envelope,
    )
    ax.plot(ts, ps.get_samples(cal_x_gates[k]).real, label=k)

k = "I1"
v = x_gates[k]
amp = np.median(cal_amps)
print(f"Adjusting {k} to amplitude {amp}")
cal_x_gates[k] = qk.pulse.ScalableSymbolicPulse(
        pulse_type=f"{k}-calibrated",
        duration=duration_1qb,
        amp=amp,
        angle=0.0,
        limit_amplitude=False,
        envelope=v.envelope,
)
ax.plot(ts, ps.get_samples(cal_x_gates[k]).real, label=k)

k = "I2"
v = ps.zero_envelope(duration_1qb)
cal_x_gates[k] = qk.pulse.ScalableSymbolicPulse(
        pulse_type=f"{k}-calibrated",
        duration=duration_1qb,
        amp=0.0,
        angle=0.0,
        limit_amplitude=False,
        envelope=v.envelope,
)
ax.plot(ts, ps.get_samples(cal_x_gates[k]).real, label=k)

# Add legend
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')

# Calibrate the default X gates
default_gates = {}
for k in pico_a.keys():
    if "SX" in k:
        default_gates[k] = ps.gaussian_envelope(duration_1qb, dt=dt, angle=np.pi / 4)
    elif "X" in k:
        default_gates[k] = ps.gaussian_envelope(duration_1qb, dt=dt, angle=np.pi / 2)
    elif "I" in k:
        default_gates[k] = ps.zero_envelope(duration_1qb)
    else:
        raise ValueError(f"Unknown gate {k}")
default_gates["I2"] = ps.zero_envelope(duration_1qb)


In [ ]:
# Load shapes
def fname(k):
    return f"../pico-pulses/saved-pulses-2024-01-16/a_two_qubit_{k}.csv"
knames = ["default", "robust"]
pico2_a = {k: ps.from_csv(fname(k))[0] for k in knames}

# Less precise about optimization than before...
fig, ax = plt.subplots(figsize=[5, 2])
for k, v in pico2_a.items():
    ax.plot(v, label=k)
ax.legend()

# Downsample the pulse to make construction faster
cr_gates = {}
for k, v in pico2_a.items():
    cr_gates[k] = ps.scalable_symbolic_pulse(
        duration_2qb,
        ps.resample(51, v, keep_step=False),
        f"cr-{k}"
    )

# Add positive and negative rotations to gateset
cal_cr_gates = {}
cal_values = [0.125 * np.pi, 0.125 * np.pi]
for θ, (k, v) in zip(cal_values, cr_gates.items()):
    amp = ps.get_amplitude_for_rotation(θ, ps.get_samples(v).real, dt)
    cal_cr_gates[k] = {
        "+": qk.pulse.ScalableSymbolicPulse(
            pulse_type=f"{k}-calibrated+amp",
            duration=duration_2qb,
            amp=amp,
            angle=0.0,
            limit_amplitude=False,
            envelope=v.envelope,
        ),
        "-": qk.pulse.ScalableSymbolicPulse(
            pulse_type=f"{k}-calibrated-amp",
            duration=duration_2qb,
            amp=-amp,
            angle=0.0,
            limit_amplitude=False,
            envelope=v.envelope,
        ),
    }

## Build QV circuit

In [ ]:
registers = [0, 1, 2, 3]
coupling_map = [[0, 1], [1, 2], [2, 3]]

In [ ]:
qv = qk.circuit.library.QuantumVolume(num_qubits=4, depth=None, seed=0, classical_permutation=True) 

transpiled_qv = qiskit.transpile(qv, basis_gates=['ecr','sx','x','rz','id']) 
transpiled_qv.draw("mpl")#, fold=-1)

In [ ]:
mb = ps.MomentBuilder(coupling_map=coupling_map)

qv_moments = mb.build_moments(
    transpiled_qv, 
    no_end_to_end=False, 
    optimization_level=0, 
)

In [ ]:
circ = ps.build_circuit(qv_moments, registers, insert_identities=True, barriers=True)
circ.draw('mpl')

# Schedules

In [ ]:
my_moments = qv_moments

In [ ]:
# backend.configuration().control_channels[(3,0)] = [qk.pulse.channels.ControlChannel(8)]
# backend.configuration().control_channels[(0,3)] = [qk.pulse.channels.ControlChannel(9)]

# backend.configuration().control_channels[(2,0)] = [qk.pulse.channels.ControlChannel(10)]
# backend.configuration().control_channels[(0,2)] = [qk.pulse.channels.ControlChannel(11)]

# backend.configuration().control_channels[(3,1)] = [qk.pulse.channels.ControlChannel(12)]
# backend.configuration().control_channels[(1,3)] = [qk.pulse.channels.ControlChannel(13)]

In [ ]:
sched = ps.build_pulse_schedule(
    my_moments,
    qubit_gates=default_gates,
    cr_gates=cal_cr_gates['default'],
    registers=registers,
    backend=backend,
    name="hamiltonian-simulation",
    dt=dt
)

robust_sched = ps.build_pulse_schedule(
    my_moments,
    qubit_gates=cal_x_gates,
    cr_gates=cal_cr_gates['robust'],
    registers=registers,
    backend=backend,
    name="hamiltonian-simulation",
    dt=dt
)

mix_sched = ps.build_pulse_schedule(
    my_moments,
    qubit_gates=default_gates,
    cr_gates=cal_cr_gates['robust'],
    registers=registers,
    backend=backend,
    name="hamiltonian-simulation",
    dt=dt,
    insert_identities=True
)

# Check durations
assert robust_sched.duration == sched.duration

# Would be nice to disable phase channels when plotting
# sched.draw()

# Models

In [ ]:
scale = 1.0

# Set registers above with circuit
# The rest is automatic
# ===================== 

# NOTE: If the Rabi rates are True, you have to calibrate!
cvars = ps.backend_simulation_vars(backend, rabi=False, units=Units)

# Change T1 and T2 to match Eagle processors like Sherbrooke
for k, v in cvars.items():
    if "t1" in k:
        cvars[k] = 250 / μs * ns
    if "t2" in k:
        cvars[k] = 150 / μs * ns

# Carrier frequencies of each control line
# NOTE: We won't use carriers in this example
carriers = ps.backend_carriers(backend, cvars)

# Partially compile model functions with parameters
# =====
H_rx = functools.partial(
    ps.rx_model,
    registers=registers,
    backend=backend,
    variables=cvars,
    rotating_frame=True,
)

H_rz = functools.partial(
    ps.rz_model,
    registers=registers,
    backend=backend,
)

H_cr = functools.partial(
    ps.cross_resonance_model,
    registers=registers,
    backend=backend,
    variables=cvars,
    model_name="Toy",
)

A_decay = functools.partial(ps.qubit_decay_model, registers=registers, variables=cvars)

edges = ps.backend_edges(backend) #[1:-1]
print(f"Edges: {edges}")
H_xtalk = scale * ps.crosstalk_model(registers, edges, cvars)

# Create a system model
# =====
H_drift = 0.0
Hs_control = []
Hs_channels = []
As_static = []
for qubit in registers:
    # Single qubit terms
    Hj_drift, Hjs_control, Hjs_channel = H_rx(qubit)
    Hj_phase, Hj_phase_ch = H_rz(qubit)
    Ajs_static = A_decay(qubit)

    H_drift += Hj_drift
    Hs_control += Hjs_control + [Hj_phase]
    Hs_channels += Hjs_channel + [Hj_phase_ch]
    As_static += Ajs_static

    # Two qubit with ring connectivity
    if qubit < len(registers) - 1:
        control, target = qubit, qubit + 1
        
        # Get both directions
        for pair in [(control, target), (target, control)]:
            Hj_drift, Hjs_control, Hjs_channel = H_cr(pair)
            H_drift += Hj_drift
            Hs_control += Hjs_control
            Hs_channels += Hjs_channel

# Some models may repeat control channels
# NOTE: This actually really matters! Repeat channels mess things up.
Hs_channels, indices = np.unique(Hs_channels, return_index=True)
Hs_control = [Hs_control[i] for i in indices]

# For RWA / Rotating frame
# NOTE: We won't use carriers in this example
# =====
registered_carriers = {c: 0 for c in Hs_channels}
cutoff_freq = None

# registered_carriers = {c: carriers[c] for c in Hs_channels}
# cutoff_freq = 2 * np.mean(list(registered_carriers.values()))

# Solvers

In [ ]:
# Models already in the rotating frame
# =====

solver_r0 = qk_d.Solver(
    static_hamiltonian=None,
    hamiltonian_operators=Hs_control,
    hamiltonian_channels=Hs_channels,
    # static_dissipators=As_static,
    rotating_frame=None,
    rwa_cutoff_freq=None,
    channel_carrier_freqs=registered_carriers,
    dt=dt
)

solver_r = qk_d.Solver(
    static_hamiltonian=None,
    hamiltonian_operators=Hs_control,
    hamiltonian_channels=Hs_channels,
    static_dissipators=As_static,
    rotating_frame=None,
    rwa_cutoff_freq=None,
    channel_carrier_freqs=registered_carriers,
    dt=dt
)

solver_rx0 = qk_d.Solver(
    static_hamiltonian=H_xtalk,
    hamiltonian_operators=Hs_control,
    hamiltonian_channels=Hs_channels,
    # static_dissipators=As_static,
    rotating_frame=None,
    rwa_cutoff_freq=None,
    channel_carrier_freqs=registered_carriers,
    dt=dt
)

solver_rx = qk_d.Solver(
    static_hamiltonian=H_xtalk,
    hamiltonian_operators=Hs_control,
    hamiltonian_channels=Hs_channels,
    static_dissipators=As_static,
    rotating_frame=None,
    rwa_cutoff_freq=None,
    channel_carrier_freqs=registered_carriers,
    dt=dt
)

# Unitary simulations

In [ ]:
# measure after each non-phase moment
measure_times = dt * np.cumsum(
    ps.durations_of_moments(my_moments, cal_x_gates, cal_cr_gates['default'])
)[[ps.get_moment_case(m) != "phase" for m in my_moments]]
# and at the end time...
measure_times = np.append(measure_times, sched.duration * dt)


# targets at measure times
Uts = []
for i, m in enumerate(my_moments):
    if ps.get_moment_case(m) != "phase":
        Uts.append(ps.build_operator(my_moments[0:i+1], registers))
# and at the end time...
Uts.append(qk.quantum_info.Operator(ps.build_circuit(my_moments, registers)))

# Target
target = Uts[-1]

In [ ]:
# # measure after each 2qb moment
# measure_times = dt * np.cumsum(
#     ps.durations_of_moments(my_moments, cal_x_gates, cal_cr_gates['default'])
# )[[ps.get_moment_case(m) == "ecr" for m in my_moments]]


# # targets at measure times
# Uts = []
# for i, m in enumerate(my_moments):
#     if ps.get_moment_case(m) == "ecr":
#         Uts.append(ps.build_operator(my_moments[0:i+1], registers))

In [ ]:
print("Case 2: Crosstalk, no decay")

y0 = ps.qiskit_identity_operator(len(registers))

# labels = ["robust (rotary)"]
# all_schedules = [robust_sched]

# labels = ["robust (rotary)",  "default (rotary)"]
# all_schedules = [robust_sched, mix_sched]

labels = ["default", "robust (rotary)", "default (rotary)"]
all_schedules = [sched, robust_sched, mix_sched]


# ==========

solver_rx0.model.evaluation_mode = "sparse"
res_x0 = solver_rx0.solve(
    t_span=[0.0, sched.duration * dt],
    t_eval=measure_times,
    y0=y0,
    signals=all_schedules,
    atol=1e-9,
    rtol=1e-9,
    method="jax_odeint",
)

# Compute fidelity (we can use our circuit to compute the target)
target = qk.quantum_info.Operator(ps.build_circuit(my_moments, registers))

for i, ell in enumerate(labels):
    print(f"\tInfidelity of {ell}: ", 1 - qk.quantum_info.process_fidelity(res_x0[i].y[-1], target))

In [ ]:
fig, axes = plt.subplots(1, figsize=[7, 4], dpi=300)
axes = axes if isinstance(axes, np.ndarray) else [axes]
# axes[0].set_title("Hamiltonian simulation: Crosstalk, No decoherence", loc='left')


for iax in range(1):
    ax = axes[iax]

    for i, res in enumerate(res_x0):
        infids = [0]
        # if labels[i] == "default":
        #     continue
        for r, t in zip(res.y, Uts):
            infids.append(1 - qk.quantum_info.process_fidelity(qk.quantum_info.Operator(r), qk.quantum_info.Operator(t)))
        ax.plot(np.hstack([0, *measure_times]), infids, label=labels[i], marker=".")

    # ax.axhline(0.1, color="black", linestyle="-", lw=0.5)
    # ax.axhline(0.01, color="black", linestyle="-", lw=0.5)
    # ax.axhline(0.001, color="black", linestyle="-", lw=0.5)
    # ax.axhline(0.0001, color="black", linestyle="-", lw=0.5)
    # # ax.axhline(1e-10, color="black", linestyle="-", lw=0.5)


    ax.set_xlabel("Time")
    if iax == 0:
        ax.legend(loc="lower right")
        ax.set_yscale("log")
        # ax.set_ylim([0, .001])
        ax.set_ylabel("Infidelity")
    else:
        ax.set_ylabel("Infidelity")
        ax.legend(loc="upper left")
        ax.set_yscale("linear")



In [ ]:
circ.draw('mpl')

### Plots for layers of Hamiltonian simulation

This is the crosstalk-free sim and plot. We won't need that sim for now.

In [ ]:
print("Case 1: No crosstalk, no decay")

y0 = ps.qiskit_identity_operator(len(registers))

solver_r0.model.evaluation_mode = "sparse"
res_0 = solver_r0.solve(
    t_span=[0.0, sched.duration * dt],
    t_eval=[sched.duration * dt],
    y0=y0,
    signals=[sched, robust_sched, mix_sched],
    atol=1e-9,
    rtol=1e-9,
    method="jax_odeint",
)

# Compute fidelity (we can use our circuit to compute the target)
target = ps.build_operator(my_moments, registers)

print(
    "\tInfidelity of default schedule: ",
    1 - qk.quantum_info.process_fidelity(res_0[0].y[-1], target),
)
print(
    "\tInfidelity of robust schedule: ",
    1 - qk.quantum_info.process_fidelity(res_0[1].y[-1], target),
)
print(
    "\tInfidelity of mixed schedule: ",
    1 - qk.quantum_info.process_fidelity(res_0[2].y[-1], target),
)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=[10, 4])
fig.suptitle("Hamiltonian Simulation: No crosstalk, No Decoherence")

n = 10
layers = np.arange(1, n + 1)
Uts = list(itertools.accumulate(np.repeat([target], n, axis=0), np.matmul))

for iax in range(2):
    ax = axes[iax]

    for i, r in enumerate(res_0):
        Ufs = itertools.accumulate(np.repeat([r.y[-1]], n, axis=0), np.matmul)
        infids = []
        for r, t in zip(Ufs, Uts):
            infids.append(1 - qk.quantum_info.process_fidelity(qk.quantum_info.Operator(r), qk.quantum_info.Operator(t)))
        ax.plot(layers, infids, label=labels[i], marker=".")

    ax.axhline(0.1, color="black", linestyle="-", lw=0.5)
    ax.axhline(0.01, color="black", linestyle="-", lw=0.5)
    ax.axhline(0.001, color="black", linestyle="-", lw=0.5)

    ax.set_xlabel("Layers")
    if iax == 0:
        ax.legend(loc="lower right")
        ax.set_yscale("log")
        ax.set_ylabel("Infidelity")
    else:
        ax.set_ylabel("Infidelity")
        ax.legend(loc="upper left")
        ax.set_yscale("linear")

This is the plot for the crosstalk example. We're still using that simulation up above.

In [ ]:
n = 100

fig, axes = plt.subplots(1, figsize=[7, 4], dpi=300)
axes = axes if isinstance(axes, np.ndarray) else [axes]
# fig.suptitle("Hamiltonian simulation")
axes[0].set_title("Hamiltonian simulation: Crosstalk, No decoherence", loc='left')

# NOTE: Manually add a layer 0 for the initial state
layers = np.arange(0, n + 1)
Ulayers = list(itertools.accumulate(np.repeat([target], n, axis=0), np.matmul))

for iax in range(1):
    ax = axes[iax]

    for i, res in enumerate(res_x0):
        # if labels[i] == "default":
        #     continue

        Ufs = itertools.accumulate(np.repeat([res.y[-1]], n, axis=0), np.matmul)
        infids = [0]
        for u, t in zip(Ufs, Ulayers):
            infids.append(1 - qk.quantum_info.process_fidelity(qk.quantum_info.Operator(u), qk.quantum_info.Operator(t)))
        # Labels set previously
        ax.plot(layers, infids, label=labels[i], marker=".")

    ax.axhline(0.1, color="black", linestyle="-", lw=0.5)
    ax.axhline(0.01, color="black", linestyle="-", lw=0.5)
    ax.axhline(0.001, color="black", linestyle="-", lw=0.5)

    ax.set_xlabel("Layers")
    if iax == 0:
        ax.legend(loc="lower right")
        ax.set_yscale("log")
        ax.set_ylabel("Infidelity")
    else:
        ax.set_ylabel("Infidelity")
        ax.legend(loc="upper left")
        ax.set_yscale("linear")

### With decoherence

In [ ]:
print("Case 3: Crosstalk and Decoherence")

# y0 = ps.qiskit_identity_operator(len(registers), open=True)
y0 = ps.qiskit_ground_state(len(registers), open=True)

labels = ["default", "robust (rotary)", "default (rotary)"]
schedules = [sched, robust_sched, mix_sched]

solver_rx.model.evaluation_mode = "sparse_vectorized"
res_x = solver_rx.solve(
    t_span=[0.0, sched.duration * dt],
    t_eval=measure_times,
    y0=y0,
    signals=schedules,
    atol=1e-9,
    rtol=1e-9,
    method="jax_odeint",
)

# Compute fidelity (we can use our circuit to compute the target)
target = qk.quantum_info.Operator(ps.build_circuit(my_moments, registers))
y_target = y0.evolve(target)

# for i in range(len(res_x)):
#     print(
#         f"\tInfidelity of {labels[i]} schedule: {1 - qk.quantum_info.process_fidelity(res_x[0].y[-1], target)}"
#     )

for i in range(len(res_x)):
    print(
        f"\tInfidelity of {labels[i]} schedule: {1 - qk.quantum_info.state_fidelity(res_x[i].y[-1], y_target)}"
    )

In [ ]:
fig, axes = plt.subplots(1, figsize=[7, 4], dpi=300)
axes = axes if isinstance(axes, np.ndarray) else [axes]
# axes[0].set_title("Hamiltonian simulation: Crosstalk, No decoherence", loc='left')


for iax in range(1):
    ax = axes[iax]

    for i, res in enumerate(res_x):
        infids = [0]
        # if labels[i] == "default":
        #     continue
        for r, t in zip(res.y, Uts):
            infids.append(1 - qk.quantum_info.process_fidelity(qk.quantum_info.Operator(r), qk.quantum_info.Operator(t)))
        ax.plot(np.hstack([0, *measure_times]), infids, label=labels[i], marker=".")

    ax.axhline(0.1, color="black", linestyle="-", lw=0.5)
    ax.axhline(0.01, color="black", linestyle="-", lw=0.5)
    ax.axhline(0.001, color="black", linestyle="-", lw=0.5)
    ax.axhline(0.0001, color="black", linestyle="-", lw=0.5)
    # ax.axhline(1e-10, color="black", linestyle="-", lw=0.5)


    ax.set_xlabel("Time")
    if iax == 0:
        ax.legend(loc="lower right")
        ax.set_yscale("log")
        # ax.set_ylim([0, .001])
        ax.set_ylabel("Infidelity")
    else:
        ax.set_ylabel("Infidelity")
        ax.legend(loc="upper left")
        ax.set_yscale("linear")

In [ ]:
print("Case 3.5: Decoherence")

# y0 = ps.qiskit_identity_operator(len(registers), open=True)
y0 = ps.qiskit_ground_state(len(registers), open=True)

labels = ["decoherence limit, default", "decoherence limit, robust (rotary)", "decoherence limit, default (rotary)"]
schedules = [sched, robust_sched, mix_sched]

solver_r.model.evaluation_mode = "sparse_vectorized"
res_r = solver_r.solve(
    t_span=[0.0, sched.duration * dt],
    t_eval=measure_times,
    y0=y0,
    signals=schedules,
    atol=1e-9,
    rtol=1e-9,
    method="jax_odeint",
)

# Compute fidelity (we can use our circuit to compute the target)
target = qk.quantum_info.Operator(ps.build_circuit(my_moments, registers))
y_target = y0.evolve(target)

for i in range(len(res_r)):
    print(
        f"\tInfidelity of {labels[i]} schedule: {1 - qk.quantum_info.state_fidelity(res_r[i].y[-1], y_target)}"
    )

In [ ]:
# Save the results
# format datetime for a file name
now = datetime.now()
dt_string = now.strftime("%Y-%m-%d-%H-%M-%S")
pkl.dump(
    {
        "res_x0": res_x0,
        "res_x": res_x,
        "res_r": res_r,
        "measure_times": measure_times,
        "Uts": Uts,
    },
    open(f"{dt_string}-scale-{scale}-crosstalk-decoherence.pkl", "wb"),
)

In [ ]:
fig, axes = plt.subplots(1, figsize=[7, 4], dpi=300)
axes = axes if isinstance(axes, np.ndarray) else [axes]
axes[0].set_title("Hamiltonian simulation: Crosstalk and decoherence", loc='left')

yts = [y0.evolve(Ut) for Ut in Uts]
labels = ["default", "robust (rotary)", "default (rotary)"]
# labels = ["decoherence limit, default", "decoherence limit, robust (rotary)", "decoherence limit, default (rotary)"]

for iax in range(len(axes)):
    ax = axes[iax]
    tab10 = plt.get_cmap("tab10")
    for i, (rx, rr) in enumerate(zip(res_x, res_r)):
        infids = [0]
        infids_decoh = [0]
        for j, t in enumerate(yts):
            infids.append(1 - qk.quantum_info.state_fidelity(rx.y[j], t))
            infids_decoh.append(1 - qk.quantum_info.state_fidelity(rr.y[j], t))
        if i == 0:
            ax.plot(np.hstack([0, *measure_times]), infids_decoh, c='k', ls='--',lw=1, zorder=10, label="decoherence limit")
        ax.plot(np.hstack([0, *measure_times]), infids, c=tab10(i), label=labels[i], marker=".")

    ax.axhline(0.1, color="black", linestyle="-", lw=0.5)
    ax.axhline(0.01, color="black", linestyle="-", lw=0.5)
    ax.axhline(0.001, color="black", linestyle="-", lw=0.5)

    ax.set_xlabel("Time")
    if iax == 0:
        ax.legend(loc="lower right")
        ax.set_yscale("log")
        ax.set_ylabel("Infidelity")
    else:
        ax.set_ylabel("Infidelity")
        ax.legend(loc="upper left")
        ax.set_yscale("linear")

In [ ]:
n = 50

fig, axes = plt.subplots(1, 2, figsize=[7, 4])#, dpi=300)
axes = axes if isinstance(axes, np.ndarray) else [axes]
# fig.suptitle("Hamiltonian simulation: Crosstalk and decoherence")
axes[0].set_title("Hamiltonian simulation: Crosstalk, No decoherence", loc='left')

layers = np.arange(1, n + 1)
Uts = list(itertools.accumulate(np.repeat([target], n, axis=0), np.matmul))

for iax in range(len(axes)):
    ax = axes[iax]

    for i, r in enumerate(res_x):
        Ufs = itertools.accumulate(np.repeat([r.y[-1]], n, axis=0), np.matmul)
        infids = []
        for r, t in zip(Ufs, Uts):
            infids.append(1 - qk.quantum_info.process_fidelity(r, qk.quantum_info.Operator(t)))
        ax.plot(layers, infids, label=labels[i], marker=".")

    ax.axhline(0.1, color="black", linestyle="-", lw=0.5)
    ax.axhline(0.01, color="black", linestyle="-", lw=0.5)
    ax.axhline(0.001, color="black", linestyle="-", lw=0.5)

    ax.set_xlabel("Layers")
    if iax == 0:
        ax.legend(loc="lower right")
        ax.set_yscale("log")
        ax.set_ylabel("Infidelity")
    else:
        ax.set_ylabel("Infidelity")
        ax.legend(loc="upper left")
        ax.set_yscale("linear")

# Combo plot

In [ ]:
# Load from file
two_res_x0 = []
two_res_x = []
two_res_r = []
with open("../playground/2024-03-01-15-06-33-scale-10-crosstalk-decoherence.pkl", "rb") as f:
    data = pkl.load(f)
    two_res_x0.append(data['res_x0'])
    two_res_x.append(data['res_x'])
    two_res_r.append(data['res_r'])

with open("../playground/2024-03-01-15-36-13-scale-1.0-crosstalk-decoherence.pkl", "rb") as f:
    data = pkl.load(f)
    two_res_x0.append(data['res_x0'])
    two_res_x.append(data['res_x'])
    two_res_r.append(data['res_r'])

In [ ]:
fig, axes = plt.subplots(1, figsize=[7, 4], dpi=300)
axes = axes if isinstance(axes, np.ndarray) else [axes]
# axes[0].set_title("Hamiltonian simulation: Crosstalk, No decoherence", loc='left')


for iax in range(1):
    ax = axes[iax]
    for i in range(len(labels)):
        res1 = two_res_x0[0][i]
        res2 = two_res_x0[1][i]
        infids1 = [0]
        infids2 = [0]
        for r1, r2, t in zip(res1.y, res2.y, Uts):
            infids1.append(1 - qk.quantum_info.process_fidelity(qk.quantum_info.Operator(r1), qk.quantum_info.Operator(t)))
            infids2.append(1 - qk.quantum_info.process_fidelity(qk.quantum_info.Operator(r2), qk.quantum_info.Operator(t)))
    
        x = np.hstack([0, *measure_times])
        ax.fill_between(x, y1=infids1, y2=infids2, color=tab10(i), alpha=0.33)
        ax.plot(x, infids1, label=labels[i], marker=".", color=tab10(i))
        ax.plot(x, infids2, marker=".", color=tab10(i))

    ax.axhline(0.1, color="black", linestyle="-", lw=0.5)
    ax.axhline(0.01, color="black", linestyle="-", lw=0.5)
    ax.axhline(0.001, color="black", linestyle="-", lw=0.5)
    ax.axhline(0.0001, color="black", linestyle="-", lw=0.5)

    ax.set_xlabel("Time")
    if iax == 0:
        ax.legend(loc="lower right")
        ax.set_yscale("log")
        ax.set_ylabel("Infidelity")
    else:
        ax.set_ylabel("Infidelity")
        ax.legend(loc="upper left")
        ax.set_yscale("linear")

In [ ]:
fig, axes = plt.subplots(1, figsize=[7, 4], dpi=300)
axes = axes if isinstance(axes, np.ndarray) else [axes]
axes[0].set_title("Hamiltonian simulation: Crosstalk and decoherence", loc='left')

yts = [y0.evolve(Ut) for Ut in Uts]
labels = ["default", "robust (rotary)", "default (rotary)"]
# labels = ["decoherence limit, default", "decoherence limit, robust (rotary)", "decoherence limit, default (rotary)"]

for iax in range(len(axes)):
    ax = axes[iax]
    tab10 = plt.get_cmap("tab10")
    for i in range(len(labels)):

        infids1 = [0]
        infids2 = [0]
        infids_decoh = [0]
        for j, t in enumerate(yts):
            infids1.append(1 - qk.quantum_info.state_fidelity(two_res_x[0][i].y[j], t))
            infids2.append(1 - qk.quantum_info.state_fidelity(two_res_x[1][i].y[j], t))
            infids_decoh.append(1 - qk.quantum_info.state_fidelity(rr.y[j], t))
        if i == 0:
            ax.plot(np.hstack([0, *measure_times]), infids_decoh, c='k', ls='--',lw=1, zorder=10, label="decoherence limit")
        x = np.hstack([0, *measure_times])
        ax.fill_between(x, y1=infids1, y2=infids2, color=tab10(i), alpha=0.33)
        ax.plot(x, infids1, label=labels[i], marker=".", color=tab10(i))
        ax.plot(x, infids2, marker=".", color=tab10(i))

    ax.axhline(0.1, color="black", linestyle="-", lw=0.5)
    ax.axhline(0.01, color="black", linestyle="-", lw=0.5)
    ax.axhline(0.001, color="black", linestyle="-", lw=0.5)

    ax.set_xlabel("Time")
    if iax == 0:
        ax.legend(loc="lower right")
        ax.set_yscale("log")
        ax.set_ylabel("Infidelity")
    else:
        ax.set_ylabel("Infidelity")
        ax.legend(loc="upper left")
        ax.set_yscale("linear")